In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model

In [2]:
import pickle
import pandas as pd
import numpy as np

In [3]:
model = load_model('model.h5')
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)

with open('onehot_encoder_geo.pkl','rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('standard_scaler.pkl','rb') as file:
    standard_scaler = pickle.load(file)

In [4]:
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [5]:
# Convert dictionary to DataFrame
input_df = pd.DataFrame([input_data])

# Label Encode Gender
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

# One-Hot Encode Geography
geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']])

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# Drop Geography and concatenate encoded columns
input_df = pd.concat(
    [input_df.drop('Geography', axis=1), geo_encoded_df],
    axis=1
)

# Arrange columns in the same order as training
input_df = input_df[
    ['CreditScore',
     'Gender',
     'Age',
     'Tenure',
     'Balance',
     'NumOfProducts',
     'HasCrCard',
     'IsActiveMember',
     'EstimatedSalary',
     'Geography_France',
     'Geography_Germany',
     'Geography_Spain']
]

# Standard Scaling
input_scaled = standard_scaler.transform(input_df)

In [7]:
# Prediction
prediction = model.predict(input_scaled)

prediction_probability = prediction[0][0]

print("Probability of Exit:", prediction_probability)

if prediction_probability > 0.5:
    print("Customer is likely to Exit.")
else:
    print("Customer is likely to Stay.")


1/1 [==============================] - 0s 43ms/step
Probability of Exit: 0.04102099
Customer is likely to Stay.
